In [1]:
!pip install -U albumentations==1.4.10 albucore==0.0.12 opencv-python==4.10.0.84 --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.9/161.9 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 MB 29.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 113.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 236.0/236.0 kB 14.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
category-encoders 2.7.0 requires scikit-learn<1.6.0,>=1.0.0, but you have scikit-learn 1.7.1 which is incompatible.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
dopamine-rl 4.1.2 requires gymnasium>=1.0.0, but you have gymnasium 0.29.0 which is incompatible.
sklearn-compat 0.1.3 requires scikit-learn<1.7,>=1.2, but you have scikit-learn 1.7.1 which is incompatible.


In [2]:
import os, math, gc, warnings, random, time
from pathlib import Path
import numpy as np
from PIL import Image
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision.models.segmentation import deeplabv3_resnet50
from torchvision.models import resnet50, ResNet50_Weights

import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm.auto import tqdm
import pandas as pd

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = True
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

Device: cuda


In [4]:
VOC_ROOT   = Path("/kaggle/input/pascal-voc-2012-dataset/VOC2012_train_val/VOC2012_train_val")
IMG_ROOT   = VOC_ROOT / "JPEGImages"
GT_ROOT    = VOC_ROOT / "SegmentationClass"
SPLIT_ROOT = VOC_ROOT / "ImageSets/Segmentation"

assert IMG_ROOT.exists() and GT_ROOT.exists() and SPLIT_ROOT.exists(), "VOC folders missing."

def read_ids(p: Path):
    return [x.strip() for x in open(p, "r") if x.strip()]

train_ids = read_ids(SPLIT_ROOT / "train.txt")
val_ids   = read_ids(SPLIT_ROOT   / "val.txt")
print(f"IDs → train={len(train_ids)}, val={len(val_ids)}  (expect ~1464/1449)")

BASE_OUT   = Path("/kaggle/working/outputs")
EXP_NAME   = "E_erode3"
SEEDS_DIR  = BASE_OUT / "pseudo_masks" / "gradcam"   # Grad-CAM source masks
PSEUDO_DIR = BASE_OUT / "pseudo_masks" / EXP_NAME    # erode-3 destination masks
EXP_DIR    = BASE_OUT / EXP_NAME
CKPT_PATH  = EXP_DIR / "deeplab_binary_best.pth"
RESULTS_CSV= EXP_DIR / "voc_val_robustness.csv"
SAMPLES_DIR= EXP_DIR / "samples"

for d in [SEEDS_DIR, PSEUDO_DIR, EXP_DIR, SAMPLES_DIR, BASE_OUT]:
    d.mkdir(parents=True, exist_ok=True)

print("Saving outputs to:", EXP_DIR)

IDs → train=1464, val=1449  (expect ~1464/1449)
Saving outputs to: /kaggle/working/outputs/E_erode3


In [5]:
class CAMHelper:
    """Minimal Grad-CAM using ResNet-50 final conv."""
    def __init__(self):
        m = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1).to(DEVICE).eval()
        self.model = m
        self.feats, self.grads = [], []
        def f_hook(_, __, out): self.feats.append(out.detach())
        def b_hook(_, grad_in, grad_out): self.grads.append(grad_out[0].detach())
        self.h1 = m.layer4[-1].conv3.register_forward_hook(f_hook)
        self.h2 = m.layer4[-1].conv3.register_full_backward_hook(b_hook)
        self.pre = torchvision.transforms.Compose([
            torchvision.transforms.ToTensor(),
            torchvision.transforms.Normalize(mean=[0.485,0.456,0.406],
                                             std=[0.229,0.224,0.225])
        ])
    def __call__(self, pil_img: Image.Image):
        self.feats.clear(); self.grads.clear()
        x = self.pre(pil_img).unsqueeze(0).to(DEVICE)
        logits = self.model(x)
        cls = logits.argmax(dim=1)
        logits[0, cls].backward()
        A = self.feats[-1][0]            # [C,H,W]
        G = self.grads[-1][0]            # [C,H,W]
        w = G.mean(dim=(1,2))            # [C]
        cam = torch.relu((w[:,None,None]*A).sum(0))
        cam = (cam - cam.min())/(cam.max()-cam.min()+1e-6)
        return cam.detach().cpu().numpy()
    def close(self):
        self.h1.remove(); self.h2.remove()

def build_gradcam_masks(ids, out_dir: Path, th=0.30, max_items=None):
    out_dir.mkdir(parents=True, exist_ok=True)
    cam = CAMHelper()
    wrote = 0
    it = ids if max_items is None else ids[:max_items]
    for img_id in tqdm(it, desc="Grad-CAM seeds"):
        ip = IMG_ROOT/f"{img_id}.jpg"
        if not ip.exists(): ip = IMG_ROOT/f"{img_id}.jpeg"
        if not ip.exists(): continue
        img = Image.open(ip).convert("RGB")
        W,H = img.size
        cam_map = cam(img)
        cam_up  = cv2.resize(cam_map, (W,H), interpolation=cv2.INTER_LINEAR)
        mask = (cam_up >= th).astype(np.uint8)*255
        Image.fromarray(mask).save(out_dir/f"{img_id}.png")
        wrote += 1
    cam.close()
    paths = list(out_dir.glob("*.png"))
    nonempty = sum((np.array(Image.open(p))>0).any() for p in paths[:100])
    print(f"Grad-CAM wrote {wrote} masks → {out_dir}")
    print(f"num masks: {len(paths)} | non-empty in first 100: {nonempty}")


if len(list(SEEDS_DIR.glob("*.png"))) == 0:
    print("No Grad-CAM seeds found; building now…")
    build_gradcam_masks(train_ids, SEEDS_DIR, th=0.30)
else:
    print(f"Found Grad-CAM seeds: {len(list(SEEDS_DIR.glob('*.png')))} (reusing)")

No Grad-CAM seeds found; building now…


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 180MB/s] 


Grad-CAM seeds:   0%|          | 0/1464 [00:00<?, ?it/s]

Grad-CAM wrote 1464 masks → /kaggle/working/outputs/pseudo_masks/gradcam
num masks: 1464 | non-empty in first 100: 100


In [6]:
def build_erode3(src_dir: Path, dst_dir: Path, ids, kernel_radius_px=3):
    dst_dir.mkdir(parents=True, exist_ok=True)
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2*kernel_radius_px+1, 2*kernel_radius_px+1))
    wrote, missing = 0, 0
    for img_id in tqdm(ids, desc="Erode-3"):
        src = src_dir / f"{img_id}.png"
        if not src.exists():
            missing += 1
            continue
        arr = np.array(Image.open(src))
        if arr.ndim == 3: arr = arr[...,0]
        fg = (arr > 0).astype(np.uint8)          # 0/1
        ero = cv2.erode(fg, k, iterations=1)     # 0/1
        out = (ero*255).astype(np.uint8)         # 0/255
        Image.fromarray(out).save(dst_dir/f"{img_id}.png")
        wrote += 1
    print(f"erode3 → wrote {wrote} masks to {dst_dir}")
    if missing: print(f"WARNING: {missing} seeds were missing in {src_dir}")

build_erode3(SEEDS_DIR, PSEUDO_DIR, train_ids, kernel_radius_px=3)

paths = sorted(PSEUDO_DIR.glob("*.png"))
nonempty = sum((np.array(Image.open(p))>0).any() for p in paths[:100])
print(f"Pseudo (erode3) masks: {len(paths)}, non-empty in first 100: {nonempty}")
assert len(paths) >= 1400, "Not enough pseudo masks — check earlier cells."

Erode-3:   0%|          | 0/1464 [00:00<?, ?it/s]

erode3 → wrote 1464 masks to /kaggle/working/outputs/pseudo_masks/E_erode3
Pseudo (erode3) masks: 1464, non-empty in first 100: 100


In [7]:
IGNORE_IDX = 255
IMG_SIZE   = 256
BATCH_TRAIN = 8
BATCH_VAL   = 8

class VOCPseudoBinary(Dataset):
    """
    Returns:
      x: FloatTensor [3,H,W]
      g: LongTensor  [H,W] in {0,1}      (pseudo mask binarized)
      q: LongTensor  [H,W] in {0,1,255}  (GT-for-eval, 255 = ignore)
      id: str
    """
    def __init__(self, ids, img_root, pseudo_root, gt_root, train=True, size=256):
        self.ids         = ids
        self.img_root    = Path(img_root)
        self.pseudo_root = Path(pseudo_root)
        self.gt_root     = Path(gt_root)
        self.train       = bool(train)
        self.size        = int(size)

        aug_train = A.Compose([
            A.HorizontalFlip(p=0.5),
            A.RandomBrightnessContrast(p=0.2),
            A.GaussianBlur(blur_limit=(3,5), p=0.15),
            A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.05, rotate_limit=15,
                               border_mode=cv2.BORDER_CONSTANT, p=0.5),
            A.Resize(self.size, self.size),
            A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
            ToTensorV2()
        ])
        aug_val = A.Compose([
            A.Resize(self.size, self.size),
            A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
            ToTensorV2()
        ])
        self.tr = aug_train if self.train else aug_val

    def __len__(self): return len(self.ids)

    def __getitem__(self, i):
        img_id = self.ids[i]

        ip = self.img_root / f"{img_id}.jpg"
        if not ip.exists(): ip = self.img_root / f"{img_id}.jpeg"
        img = np.array(Image.open(ip).convert("RGB"))
        H,W = img.shape[:2]

        pp = self.pseudo_root / f"{img_id}.png"
        if pp.exists(): pm = np.array(Image.open(pp))
        else:           pm = np.zeros((H,W), np.uint8)

        gp = self.gt_root / f"{img_id}.png"
        if gp.exists(): gt = np.array(Image.open(gp))
        else:           gt = np.full((H,W), IGNORE_IDX, np.uint8)

        if pm.shape != (H,W): pm = cv2.resize(pm, (W,H), interpolation=cv2.INTER_NEAREST)
        if gt.shape != (H,W): gt = cv2.resize(gt, (W,H), interpolation=cv2.INTER_NEAREST)

        gbin = np.where(gt==IGNORE_IDX, IGNORE_IDX, (gt!=0).astype(np.uint8))  # 0/1/255

        out = self.tr(image=img, mask=pm, masks=[gbin])
        x   = out["image"].float()
        pm2 = out["mask"]
        gt2 = out["masks"][0]

        pm2 = torch.as_tensor(pm2).squeeze()
        if pm2.dtype.is_floating_point:
            g = (pm2 > 0.5).long()
        else:
            g = (pm2 > 0).long()

        q = torch.as_tensor(gt2).squeeze().long()
        return x, g, q, img_id

def make_loaders(num_workers=2, pin=True):
    train_ds = VOCPseudoBinary(train_ids, IMG_ROOT, PSEUDO_DIR, GT_ROOT, train=True,  size=IMG_SIZE)
    val_ds   = VOCPseudoBinary(val_ids,   IMG_ROOT, PSEUDO_DIR, GT_ROOT, train=False, size=IMG_SIZE)
    train_dl = DataLoader(train_ds, batch_size=BATCH_TRAIN, shuffle=True,
                          num_workers=num_workers, pin_memory=pin, persistent_workers=False)
    val_dl   = DataLoader(val_ds, batch_size=BATCH_VAL,   shuffle=False,
                          num_workers=num_workers, pin_memory=pin, persistent_workers=False)
    return train_dl, val_dl

train_dl, val_dl = make_loaders()

bx, bg, bq, bids = next(iter(train_dl))
print("Shapes:", tuple(bx.shape), tuple(bg.shape), tuple(bq.shape),
      "| batches →", len(train_dl), len(val_dl))

Shapes: (8, 3, 256, 256) (8, 256, 256) (8, 256, 256) | batches → 183 182


In [8]:
def build_deeplab_binary(num_classes=2):
    # ImageNet backbone when available; otherwise random init
    try:
        m = deeplabv3_resnet50(weights_backbone="IMAGENET1K_V1")
    except Exception:
        m = deeplabv3_resnet50(weights_backbone=None)
    # replace classifier head for 2 classes
    m.classifier[-1] = nn.Conv2d(m.classifier[-1].in_channels, num_classes, kernel_size=1)
    return m

model = build_deeplab_binary().to(DEVICE)
criterion = nn.CrossEntropyLoss(ignore_index=IGNORE_IDX)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type=="cuda"))

In [10]:
EPOCHS  = 15
PATIENCE= 5

@torch.no_grad()
def evaluate(val_loader):
    model.eval()
    iou_bg, iou_fg, miou = [], [], []
    for x,g,q,_ in val_loader:
        x,g,q = x.to(DEVICE), g.to(DEVICE), q.to(DEVICE)
        logits = model(x)["out"]                       # [B,2,H,W]
        pred = logits.argmax(1)                        # [B,H,W] {0,1}
        valid = (q != IGNORE_IDX)
        # IoU for bg=0, fg=1 separately
        for cls in [0,1]:
            inter = ((pred==cls) & (q==cls) & valid).sum().item()
            union = (((pred==cls) | (q==cls)) & valid).sum().item()
            iou = inter/union if union>0 else 0.0
            (iou_bg if cls==0 else iou_fg).append(iou)
        miou.append(0.5*(iou_bg[-1]+iou_fg[-1]))
    return {
        "iou_bg": float(np.mean(iou_bg)),
        "iou_fg": float(np.mean(iou_fg)),
        "miou":   float(np.mean(miou)),
    }

def train_one_epoch():
    model.train()
    total = 0.0
    pbar = tqdm(train_dl, desc="Train", leave=False)
    for x,g,q,_ in pbar:
        x,g,q = x.to(DEVICE), g.to(DEVICE), q.to(DEVICE)
        with torch.cuda.amp.autocast(enabled=(DEVICE.type=="cuda")):
            logits = model(x)["out"]
            loss   = criterion(logits, g)             # supervise on pseudo g (0/1)
        scaler.scale(loss).backward()
        scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
        total += loss.item() * x.size(0)
        pbar.set_postfix(loss=f"{loss.item():.4f}")
    return total/len(train_dl.dataset)

best_miou, wait = -1.0, 0
for ep in range(1, EPOCHS+1):
    t0=time.time()
    loss = train_one_epoch()
    mets = evaluate(val_dl)
    print(f"Epoch {ep:02d} | loss={loss:.4f} | mIoU={mets['miou']:.3f} (bg={mets['iou_bg']:.3f}, fg={mets['iou_fg']:.3f}) | lr={opt.param_groups[0]['lr']:.2e} | {time.time()-t0:.1f}s")
    if mets["miou"] > best_miou + 1e-4:
        best_miou = mets["miou"]; wait = 0
        torch.save(model.state_dict(), CKPT_PATH)
        print("  New best; checkpoint saved ", CKPT_PATH.name)
    else:
        wait += 1
        if wait > PATIENCE:
            print("Early stop # ")
            break

print("Best mIoU:", round(best_miou, 3))

Train:   0%|          | 0/183 [00:00<?, ?it/s]

Epoch 01 | loss=0.4293 | mIoU=0.524 (bg=0.735, fg=0.314) | lr=1.00e-03 | 62.5s
  New best; checkpoint saved  deeplab_binary_best.pth


Train:   0%|          | 0/183 [00:00<?, ?it/s]

Epoch 02 | loss=0.4270 | mIoU=0.515 (bg=0.722, fg=0.308) | lr=1.00e-03 | 64.5s


Train:   0%|          | 0/183 [00:00<?, ?it/s]

...

Epoch 05 | loss=0.4236 | mIoU=0.508 (bg=0.721, fg=0.295) | lr=1.00e-03 | 68.1s


Train:   0%|          | 0/183 [00:00<?, ?it/s]

Epoch 06 | loss=0.4218 | mIoU=0.507 (bg=0.732, fg=0.282) | lr=1.00e-03 | 68.3s


Train:   0%|          | 0/183 [00:00<?, ?it/s]

Epoch 07 | loss=0.4194 | mIoU=0.494 (bg=0.719, fg=0.270) | lr=1.00e-03 | 68.6s
Early stop # 
Best mIoU: 0.524


In [11]:
def perturbations(img):
    out = {'clean': img}
    out['blur']       = cv2.GaussianBlur(img, (5,5), 1.0)
    out['brightness'] = np.clip(img.astype(np.float32)*1.15, 0, 255).astype(np.uint8)
    out['gauss']      = np.clip(img.astype(np.float32) + np.random.normal(0, 8, img.shape), 0, 255).astype(np.uint8)
    out['hflip']      = img[:, ::-1, :]
    out['rotation']   = cv2.warpAffine(img, cv2.getRotationMatrix2D((img.shape[1]//2, img.shape[0]//2), 10, 1.0),
                                       (img.shape[1], img.shape[0]), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)
    return out

def eval_under_perturbations(val_ids, max_items=None):
    model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
    model.eval()
    rows=[]
    it = val_ids if max_items is None else val_ids[:max_items]
    pbar = tqdm(it, desc="Robustness", leave=True)
    for img_id in pbar:
        ip = IMG_ROOT/f"{img_id}.jpg"
        if not ip.exists(): ip = IMG_ROOT/f"{img_id}.jpeg"
        img = np.array(Image.open(ip).convert("RGB"))
        gt_p = GT_ROOT/f"{img_id}.png"
        gt = np.array(Image.open(gt_p)) if gt_p.exists() else np.full(img.shape[:2], IGNORE_IDX, np.uint8)
        gbin = np.where(gt==IGNORE_IDX, IGNORE_IDX, (gt!=0).astype(np.uint8))

        for name, arr in perturbations(img).items():
            arr = cv2.resize(arr, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)
            gtt = cv2.resize(gbin, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)

            x = A.Compose([
                A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
                ToTensorV2()
            ])(image=arr)["image"].unsqueeze(0).to(DEVICE)

            with torch.no_grad():
                pred = model(x)["out"].argmax(1)[0].cpu().numpy()

            valid = (gtt != IGNORE_IDX)
            iou_bg = 0.0; iou_fg = 0.0
            for cls,acc in [(0,'bg'),(1,'fg')]:
                inter = np.logical_and(pred==cls, gtt==cls) & valid
                union = np.logical_or(pred==cls, gtt==cls) & valid
                val = inter.sum()/max(1, union.sum())
                if acc=='bg': iou_bg = val
                else: iou_fg = val
            miou = 0.5*(iou_bg + iou_fg)
            rows.append((name, iou_bg, iou_fg, miou))

    df = pd.DataFrame(rows, columns=["perturb","IoU_bg","IoU_fg","mIoU"])
    df = df.groupby("perturb", as_index=False).mean().sort_values("perturb").reset_index(drop=True)
    EXP_DIR.mkdir(parents=True, exist_ok=True)
    df.to_csv(RESULTS_CSV, index=False)
    print("Saved:", RESULTS_CSV)
    return df

df_rob = eval_under_perturbations(val_ids, max_items=None)
df_rob

Robustness:   0%|          | 0/1449 [00:00<?, ?it/s]

Saved: /kaggle/working/outputs/E_erode3/voc_val_robustness.csv


,perturb,IoU_bg,IoU_fg,mIoU
0,blur,0.731165,0.308748,0.519956
1,brightness,0.728095,0.321251,0.524673
2,clean,0.729910,0.314756,0.522333
3,gauss,0.729315,0.319081,0.524198
4,hflip,0.697760,0.246168,0.471964
5,rotation,0.723882,0.307425,0.515653
